# Group project · the animation template

Your animation is a small web page that plays your simulation like a film: the supplier, the transport, the stock of your key component, the stations with their queues, and the finished units, week by week. The page reads all its numbers from one file, `results.json`, which your model notebook writes. A JSON file is a plain text file that stores nested lists and dictionaries, the same structures you use in Python, so other programs can read them.

The notebook has three steps:

1. Play the template on the UrbanRide example.
2. See which numbers `results.json` needs.
3. Write your own `results.json` and save your `animation.html` (the file you hand in).

Choose *Runtime → Run all* once, then work through the steps.

## Step 1 · Play the example

In [ ]:
import os, json
if not os.path.exists("sscm-2026-student"):
    !git clone -q https://github.com/TH-Koln-Bartnik/sscm-2026-student
PROJECT = "sscm-2026-student/week04/project"
example = json.load(open(f"{PROJECT}/example_results.json"))
print(example["product"], "·", list(example["scenarios"]))

In [ ]:
from IPython.display import HTML, display

REQUIRED_WEEKLY = ["queue", "in_transit", "stock", "finished", "stopped_hours"]
REQUIRED_SCORE = ["units_per_week", "cost_per_unit_eur", "kg_co2e_per_unit", "hours_stopped"]

def make_animation(results, path="animation.html", show=True):
    """Put `results` into the template, save it as `path` and show it here."""
    for s in ["baseline", "disruption", "countermeasure"]:
        sc = results["scenarios"][s]                      # KeyError = scenario missing
        for k in REQUIRED_SCORE:
            assert k in sc["scoreboard"], f"{s}: scoreboard lacks {k}"
        for k in REQUIRED_WEEKLY:
            assert k in sc["weeks"], f"{s}: weeks lacks {k}"
    template = open(f"{PROJECT}/animation_template.html", encoding="utf-8").read()
    html = template.replace("/*RESULTS_JSON*/null", json.dumps(results))
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"saved {path}: open it in a browser, or download it from the file panel on the left")
    if show:
        display(HTML(html))

make_animation(example, path="example_animation.html")

**Try it:** press *Play*, switch to the disruption and drag the slider to week 23. Where does the stock run out? Which station still has parts waiting while the line stands still? Then switch to the countermeasure: how many weeks of stoppage does the safety stock save?

The example numbers come from a simple weekly model of the bike line and are for illustration only.

## Step 2 · What `results.json` must contain

One dictionary, with one block per scenario. Every list under `weeks` has one entry per simulated week (52), in week order. If you ran 10 seeds, show one representative seed (for example seed 1) in `weeks` and put the **means over the 10 seeds** in `scoreboard`; the poster reports the range.

| Key | What it is |
|---|---|
| `product`, `unit` | product name; what you count (`"chairs"`, `"kettles"`) |
| `key_component`, `supplier`, `transport` | text for the labels |
| `stations` | station names in line order (as in workcenters.csv) |
| `scenarios` → `baseline`, `disruption`, `countermeasure` | one block each, with `label` (button text), `scoreboard` and `weeks` |
| `scoreboard` | `units_per_week`, `cost_per_unit_eur`, `kg_co2e_per_unit`, `hours_stopped` |
| `weeks` → `queue` | per week a list with one number per station: units waiting in front of it (week average or week end, say which on the poster) |
| `weeks` → `in_transit`, `stock` | key components on their way; key components in stock at the plant |
| `weeks` → `finished`, `stopped_hours` | units finished in the week; hours the line stood still for lack of parts |
| `weeks` → `supplier_down` | optional: `True` in the weeks your supplier does not deliver |

Look at one block of the example:

In [ ]:
b = example["scenarios"]["disruption"]
print(b["label"], b["scoreboard"])
for k, v in b["weeks"].items():
    print(f"{k:14s} weeks 20-24: {v[19:24]}")

## Step 3 · Your own animation

Your model collects one list per weekly number while it runs, for each scenario. The cell below shows the shape; replace the example lists with your own. When all three scenarios are filled in, the last line saves `results.json` and `animation.html`.

In [ ]:
def scenario_block(label, scoreboard, queue, in_transit, stock, finished, stopped_hours, supplier_down=None):
    """Pack one scenario's numbers in the format the template reads."""
    weeks = {"queue": queue, "in_transit": in_transit, "stock": stock,
             "finished": finished, "stopped_hours": stopped_hours}
    if supplier_down is not None:
        weeks["supplier_down"] = supplier_down
    return {"label": label, "scoreboard": scoreboard, "weeks": weeks}

# ✏️ replace the example values with your model's results
ex = example["scenarios"]
my_results = {
    "product": example["product"],            # e.g. "Ergo office chair"
    "unit": example["unit"],                  # e.g. "chairs"
    "key_component": example["key_component"],
    "supplier": example["supplier"],
    "transport": example["transport"],
    "stations": example["stations"],          # your station names, in line order
    "scenarios": {
        s: scenario_block(ex[s]["label"], ex[s]["scoreboard"], **ex[s]["weeks"])
        for s in ["baseline", "disruption", "countermeasure"]
    },
}

with open("results.json", "w") as f:
    json.dump(my_results, f)
make_animation(my_results, path="animation.html")

**Restyling.** You may change colours, layout or labels, or rebuild the page in D3 (a JavaScript library for data-driven graphics). Download `animation_template.html` from the [project folder](https://github.com/TH-Koln-Bartnik/sscm-2026-student/tree/main/week04/project) and edit a copy. The rule from the brief stays: every number on the page comes from `results.json`, and the viewer can switch between the three scenarios.

_Prepared with Claude (Anthropic) as a coding assistant; checked and released by Professor Bartnik._